In [1]:
import pandas as pd
from pathlib import Path
import sys
sys.path.append("../src")
from utils import get_season

DF_PATH = Path("../data/processed/merged_df.parquet")
df = pd.read_parquet(DF_PATH)

In [2]:
# look at the dataframe
df.head()

,Store,DayOfWeek,Date,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Sales
0,1,5,2015-07-31,555,1,1,0,1,c,a,1270,9,2008,0,0,0,None,5263
1,2,5,2015-07-31,625,1,1,0,1,a,a,570,11,2007,1,13,2010,"Jan,Apr,Jul,Oct",6064
2,3,5,2015-07-31,821,1,1,0,1,a,a,14130,12,2006,1,14,2011,"Jan,Apr,Jul,Oct",8314
3,4,5,2015-07-31,1498,1,1,0,1,c,c,620,9,2009,0,0,0,None,13995
4,5,5,2015-07-31,559,1,1,0,1,a,a,29910,4,2015,0,0,0,None,4822


In [3]:
# date features
df["Year"] = df.Date.dt.year
df["Month"] = df.Date.dt.month
df["Day"] = df.Date.dt.day
df["WeekOfYear"] = df.Date.dt.isocalendar().week.astype("int8")
df["Quarter"] = df.Date.dt.quarter.astype("int8")
df["IsWeekend"] = (df.Date.dt.dayofweek > 4).astype("int8") 

In [4]:
# season from Date
df["Season"] = (df.Date.map(get_season)).astype("category")

In [5]:
# does competition exist?
df["CompetitionExists"] = (df.CompetitionOpenSinceYear > 0).astype("int8")

In [6]:
# was Promo2 active on the date recorded?
promo_start = (df.Promo2SinceYear * 100 + df.Promo2SinceWeek)
current = (df.Year * 100 + df.WeekOfYear)
df["WasPromo2Active"] = ((df.Promo2 == 1) & (current >= promo_start)).astype("int8")

In [7]:
# competition age in months
competition_age = ((df.Year - df.CompetitionOpenSinceYear) * 12 + (df.Month - df.CompetitionOpenSinceMonth))
df["CompetitionAgeMonths"] = competition_age.where(df["CompetitionExists"] == 1, 0).clip(lower=0).astype("int16")

In [8]:
# drop Date
df = df.drop("Date", axis=1)

In [9]:
# move Sales target to the end
col_to_move = df.pop("Sales")
df.insert(len(df.columns), "Sales", col_to_move)

In [10]:
# look at the final feature engineered dataframe
df.head()

,Store,DayOfWeek,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,...,Month,Day,WeekOfYear,Quarter,IsWeekend,Season,CompetitionExists,WasPromo2Active,CompetitionAgeMonths,Sales
0,1,5,555,1,1,0,1,c,a,1270,...,7,31,31,3,0,Summer,1,0,82,5263
1,2,5,625,1,1,0,1,a,a,570,...,7,31,31,3,0,Summer,1,1,92,6064
2,3,5,821,1,1,0,1,a,a,14130,...,7,31,31,3,0,Summer,1,1,103,8314
3,4,5,1498,1,1,0,1,c,c,620,...,7,31,31,3,0,Summer,1,0,70,13995
4,5,5,559,1,1,0,1,a,a,29910,...,7,31,31,3,0,Summer,1,0,3,4822


In [11]:
# save feature engineered df
df.to_parquet("../data/processed/feature_engineered_df.parquet", index=False)